# Route B: BMZ BirdNET to JSONL to analysis

1. bioacoustics-model-zoo BirdNET().predict() and .embed() on audio.
2. The adapter writes a JSONL manifest with 1024-d embeddings.
3. bioacoustic-embedding-dynamics runs PCA, UMAP, trajectory, change-point, and HMM.

Lighter than bacpipe. Needs TensorFlow or ai-edge-litert for BirdNET TFLite.

## 1. Get this repo

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/path/to/bioacoustic-embedding-dynamics

import os
from pathlib import Path
print("cwd:", os.getcwd())

## 2. Install (BMZ BirdNET and this package)

In [ ]:
import sys
!{sys.executable} -m pip install -q "bioacoustics-model-zoo[birdnet]"
!{sys.executable} -m pip install -q -r docs/colab-requirements.txt
!{sys.executable} -m pip install -q -e .

## 3. Audio files

Upload `.wav` to `/content/audio/` or set `AUDIO_GLOB` to your paths.

In [ ]:
AUDIO_DIR = Path("/content/audio")
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

# Example: upload via Colab file picker, or mount Drive
# from google.colab import files
# uploaded = files.upload()
# for name, data in uploaded.items():
#     (AUDIO_DIR / name).write_bytes(data)

audio_files = sorted(AUDIO_DIR.glob("*.wav")) + sorted(AUDIO_DIR.glob("*.WAV"))
if not audio_files:
    raise FileNotFoundError(
        f"No .wav in {AUDIO_DIR}. Upload field recordings before running Route B."
    )
print(f"Found {len(audio_files)} file(s)")
for p in audio_files[:5]:
    print(" -", p.name)

## 4. BMZ BirdNET to JSONL

In [ ]:
from bioacoustic_embedding_dynamics.adapters import bmz_birdnet_to_manifest

MANIFEST = Path("data/bmz_birdnet.jsonl")
bmz_birdnet_to_manifest(
    audio_files,
    MANIFEST,
    batch_size=32,
    min_confidence=0.001,
)
print(f"Wrote {MANIFEST} ({sum(1 for _ in MANIFEST.open())} lines)")

## 5. Run embedding analysis

In [ ]:
!python -m bioacoustic_embedding_dynamics.cli --manifest {MANIFEST} --out reports/bmz --seed 42

## 6. Summary and figures

In [ ]:
import json
from IPython.display import Image, display

summary = json.loads(Path("reports/bmz/summary.json").read_text())
print(json.dumps(summary, indent=2))

for name in [
    "pca_species.png", "umap_species.png", "trajectory_pca.png",
    "changepoints.png", "trajectory_changepoints.png", "hmm_regimes.png",
]:
    p = Path("reports/bmz") / name
    if p.is_file():
        display(Image(filename=str(p)))